In [1]:
from torch.nn.functional import mse_loss

from LIDBagging.experiment_class import *
from LIDBagging.RunningEstimators.BaggingSmoothing.SimpleBagging import *

In [ ]:
int(1.12)

In [2]:
from tqdm.notebook import tqdm
import os, structlog
import matplotlib.pyplot as plt

import os, structlog
_devnull = open(os.devnull, "w")
structlog.configure(logger_factory=structlog.PrintLoggerFactory(file=_devnull))

In [3]:
test_exp_skdim = LID_experiment(n=2500, k=10, sr=1, Nbag=1, dataset_name="M11_Moebius", lid=2, dim=3, pre_smooth=False,
                 post_smooth=False, t=1, estimator='mle', bagging_method='bag', submethod_0=None, submethod_error=None, param_string=None, params=None)
test_exp_skdim.generate_data(load=True)
e1, a1 = test_exp_skdim.estimate(use_LIDkit=False)

In [ ]:
test_exp_LIDkit = LID_experiment(n=2500, k=10, sr=1, Nbag=1, dataset_name="M11_Moebius", lid=2, dim=3, pre_smooth=False,
                 post_smooth=False, t=1, estimator='mle', bagging_method='bag', submethod_0=None, submethod_error=None, param_string=None, params=None)
test_exp_LIDkit.generate_data(load=True)
e2, a2 = test_exp_LIDkit.estimate(use_LIDkit=True)

KeyboardInterrupt: 

In [6]:
from scipy.spatial.distance import pdist, squareform

In [13]:
import numpy as np
#from torch import log_
try:
    import faiss
except ImportError:
    faiss = None

In [14]:
index = None
IndexFlatL2 = getattr(faiss, 'IndexFlatL2', None)
N, D = test_exp_LIDkit.data[0].shape
index = IndexFlatL2(D)

In [54]:
dist1 = squareform(pdist(test_exp_LIDkit.data[0]))

In [57]:
dist1.shape

(2500, 2500)

In [34]:
list1 = [5.9604645e-08, 3.7407875e-03, 5.7110190e-03, 6.6816211e-03,
       1.6251862e-02, 2.0594478e-02, 2.4624467e-02, 2.4724722e-02,
       2.9412687e-02, 3.1865597e-02, 3.4438491e-02]

In [59]:
list2 = dist1[4]

In [60]:
np.sort(list2)

array([0.        , 0.06116222, 0.07557119, ..., 1.99941581, 2.00237452,
       2.02982484])

In [37]:
list1

[5.9604645e-08,
 0.0037407875,
 0.005711019,
 0.0066816211,
 0.016251862,
 0.020594478,
 0.024624467,
 0.024724722,
 0.029412687,
 0.031865597,
 0.034438491]

In [11]:
np.min(dist1[4][dist1[4] != 0])

0.0

In [3]:
def large_test(trials = 10, srrange=10, datasetname = "uniform"):
    srs = [(srrange-i)/srrange for i in range(srrange)]
    msed = {sr: None for sr in srs}
    bias1d = {sr: None for sr in srs}
    bias2d = {sr: None for sr in srs}
    yd = {sr: None for sr in srs}
    zd = {sr: None for sr in srs}
    for sr in srs:
        mse = np.empty(trials)
        bias1 = np.empty(trials)
        bias2 = np.empty(trials)
        y = np.empty(trials)
        z = np.empty(trials)
        for trial in tqdm(range(trials)):
            
            test_exp_skdim = LID_experiment(n=2500, k=10, sr=sr, Nbag=10, dataset_name=datasetname, lid=2, dim=3, pre_smooth=False,
                     post_smooth=False, t=1, estimator='mle', bagging_method='bag', submethod_0=None, submethod_error=None, param_string=None, params=None)
            test_exp_skdim.generate_data(load=True)
            e1, a1 = test_exp_skdim.estimate(use_LIDkit=False)
            
            test_exp_LIDkit = LID_experiment(n=2500, k=10, sr=sr, Nbag=10, dataset_name=datasetname, lid=2, dim=3, pre_smooth=False,
                     post_smooth=False, t=1, estimator='mle', bagging_method='bag', submethod_0=None, submethod_error=None, param_string=None, params=None)
            test_exp_LIDkit.generate_data(load=True)
            e2, a2 = test_exp_LIDkit.estimate(use_LIDkit=True)
            
            mse[trial] = np.mean((test_exp_skdim.lid_estimates-test_exp_LIDkit.lid_estimates)**2)
            bias2[trial] = (np.mean(test_exp_skdim.lid_estimates)-np.mean(test_exp_LIDkit.lid_estimates))**2
            bias1[trial] = (np.mean(test_exp_skdim.lid_estimates)-np.mean(test_exp_LIDkit.lid_estimates))
            y[trial] = np.mean(test_exp_skdim.lid_estimates)
            z[trial] = np.mean(test_exp_LIDkit.lid_estimates)
        msed[sr] = np.mean(mse)
        bias1d[sr] = np.mean(bias1)
        bias2d[sr] = np.mean(bias2)
        yd[sr] = np.mean(y)
        zd[sr] = np.mean(z)
    return msed, bias1d, bias2d, yd, zd

In [4]:
msed, bias1d, bias2d, yd, zd = large_test(trials = 10, srrange=10, datasetname = "M11_Moebius")

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [5]:
def plot(yd=None, zd = None, suffix = ""):
    '''
    plt.plot(list(msed.keys()), list(msed.values()))
    plt.xlabel('sr')
    plt.ylabel('mse')
    plt.savefig(f'mse_{suffix}.pdf')
    plt.close()
    
    plt.plot(list(bias2d.keys()), list(bias2d.values()))
    plt.xlabel('sr')
    plt.ylabel('bias2')
    plt.savefig(f'bias2_{suffix}.pdf')
    plt.close()
    
    plt.plot(list(bias1d.keys()), list(bias1d.values()))
    plt.xlabel('sr')
    plt.ylabel('bias')
    plt.savefig(f'bias_{suffix}.pdf')
    plt.close()
    '''
    if yd is not None and zd is not None:
        plt.plot(list(yd.keys()), list(yd.values()), label='original')
        plt.plot(list(zd.keys()), list(zd.values()), label='LIDkit')
        plt.xlabel('sr')
        plt.ylabel('avgestimate')
        plt.legend()
        plt.savefig(f'avgestimates_{suffix}.pdf')
        plt.close()
        

In [6]:
plot(yd, zd, suffix="TrialAfterFix3_M11_Moebius")



In [31]:
test_exp_skdim.lid_estimates

array([3.38311172, 2.87814517, 2.25613552, ..., 1.25026084, 2.57105412,
       3.61031761])

In [32]:
test_exp_LIDkit.lid_estimates

array([3.38308  , 2.8781927, 2.2562447, ..., 1.2499896, 2.5721772,
       3.610218 ], dtype=float32)

In [33]:
np.mean(test_exp_skdim.lid_estimates)

2.475976118114569

In [34]:
np.mean(test_exp_LIDkit.lid_estimates)

2.47595